# 03 - Statistiques descriptives : le notebook pratique

> Complement pratique du cours [`01-statistiques-descriptives.md`](./01-statistiques-descriptives.md).

Tu viens d'etre embauche comme Data Analyst. On te tend un fichier de ventes et une question floue : **"Dis-moi ce qui se passe dans mes magasins."** Ce notebook applique les gestes du cours sur le **vrai** dataset `ventes_magasins.csv` :

1. **Moyenne vs mediane** (et pourquoi ce choix n'est pas un detail)
2. **Ecart-type** : les ventes sont-elles regulieres ?
3. **Quartiles et IQR** : le coeur des 50 % centraux
4. **Histogramme et boxplot** : voir la forme des donnees
5. **Detection d'outliers** (regle 1,5 x IQR + z-score)

La colonne cle est `montant` : c'est le **panier** de chaque vente (en euros).

**Le reflexe de depart :** avant tout calcul, on regarde `info()` (types + valeurs manquantes) et `describe()` (le resume statistique en une ligne). Piege des **faux nombres** : `client_id` est un `int64` mais reste une **etiquette qualitative** (pas de "client_id moyen").

In [ ]:
# Imports + chargement du VRAI dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

ventes = pd.read_csv("../../../99-Brief/Data-Analyst/data/ventes_magasins.csv")

print("Dimensions :", ventes.shape)  # (lignes, colonnes)
ventes.head()

In [ ]:
ventes.info()
print("\n--- Ventes par ville ---")
print(ventes["ville"].value_counts())
print("\n--- describe() sur les colonnes quantitatives ---")
ventes[["quantite", "prix_unitaire", "montant", "marge"]].describe().round(2)

## Etape 1 - Moyenne vs mediane (et POURQUOI)

> La regle d'or : donnees **symetriques** -> moyenne ; donnees **asymetriques ou avec outliers** (paniers, prix, salaires...) -> mediane.

**Le test express :** si `moyenne > mediane`, la distribution a une **queue a droite** (quelques gros paniers tirent la moyenne vers le haut) -> on communique la **mediane**. Pour bien *voir* le defaut de la moyenne, on injecte ensuite **un seul** panier absurde (erreur de saisie a 50 000 EUR) et on regarde qui bouge.

In [ ]:
moyenne = ventes["montant"].mean()
mediane = ventes["montant"].median()
mode = ventes["montant"].mode()[0]

print(f"Moyenne : {moyenne:.2f} EUR")
print(f"Mediane : {mediane:.2f} EUR")
print(f"Mode    : {mode:.2f} EUR")
print(f"Moyenne - Mediane = {moyenne - mediane:.2f} EUR")
if moyenne > mediane:
    print("=> Moyenne > Mediane : asymetrie a DROITE. Le panier typique a communiquer est la MEDIANE.")
else:
    print("=> Distribution ~ symetrique : la moyenne est fiable.")

# Sensibilite en direct : un seul panier errone a 50 000 EUR
paniers = ventes["montant"]
paniers_pollue = pd.concat([paniers, pd.Series([50000])], ignore_index=True)
print("\nApres ajout d'UN panier a 50 000 EUR :")
print(f"  Moyenne {paniers.mean():.2f} -> {paniers_pollue.mean():.2f}")
print(f"  Mediane {paniers.median():.2f} -> {paniers_pollue.median():.2f}")
print("  => La mediane ne bouge quasiment pas : elle est ROBUSTE.")

## Etape 2 - L'ecart-type : les ventes sont-elles regulieres ?

L'ecart-type mesure a quel point les valeurs sont serrees autour de la moyenne. **Petit** = regulier/previsible ; **grand** = volatil.

> Attention au `ddof` : **pandas** divise par `n-1` (`ddof=1`, echantillon) par defaut, **numpy** par `n` (`ddof=0`, population). Source classique d'ecarts entre resultats.

On compare aussi la regularite entre villes avec le **coefficient de variation** (`CV = ecart-type / moyenne`), qui neutralise les differences d'echelle.

In [ ]:
etendue = ventes["montant"].max() - ventes["montant"].min()
variance = ventes["montant"].var(ddof=0)
ecart_type = ventes["montant"].std(ddof=0)

print(f"Etendue (max - min) : {etendue:.2f} EUR")
print(f"Variance (ddof=0)   : {variance:.2f} (EUR au carre, non interpretable)")
print(f"Ecart-type (ddof=0) : {ecart_type:.2f} EUR (meme unite -> on communique avec ca)")

print("\n--- Regularite par ville (CV en %) ---")
resume = ventes.groupby("ville")["montant"].agg(["mean", "std"])
resume["CV_%"] = (resume["std"] / resume["mean"] * 100).round(1)
resume.round(2).sort_values("CV_%")

## Etape 3 - Quartiles et IQR : le coeur des 50 % centraux

Les quartiles decoupent la serie triee en 4 parts de 25 %. L'**IQR = Q3 - Q1** est l'etendue des **50 % centraux** (le "coeur" des clients). Il est **robuste** : il ignore les 25 % du bas et du haut, donc les extremes ne l'affectent pas.

In [ ]:
q1 = ventes["montant"].quantile(0.25)
q2 = ventes["montant"].quantile(0.50)  # = mediane
q3 = ventes["montant"].quantile(0.75)
iqr = q3 - q1

print(f"Q1 (25%)      : {q1:.2f} EUR")
print(f"Q2 (mediane)  : {q2:.2f} EUR")
print(f"Q3 (75%)      : {q3:.2f} EUR")
print(f"IQR = Q3 - Q1 : {iqr:.2f} EUR")
print(f"=> La moitie centrale des ventes se situe entre {q1:.0f} et {q3:.0f} EUR.")

# Percentiles metier : les 10 % de meilleurs paniers
print("\nPercentiles P90 / P95 (cibles marketing haut de gamme) :")
print(ventes["montant"].quantile([0.90, 0.95]).round(2))

## Etape 4 - Histogramme + boxplot : voir la forme

L'**histogramme** montre la forme detaillee d'une variable : sur un `montant` asymetrique a droite, on voit un pic a gauche et une **longue traine**. Le **boxplot** resume d'un coup la boite (Q1->Q3), la mediane, les moustaches (1,5 x IQR) et les outliers ; il est ideal pour **comparer des groupes**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme + moyenne/mediane
sns.histplot(data=ventes, x="montant", bins=40, kde=True, ax=axes[0])
axes[0].axvline(moyenne, color="red", linestyle="--", label=f"Moyenne = {moyenne:.0f}")
axes[0].axvline(mediane, color="green", linestyle="-", label=f"Mediane = {mediane:.0f}")
axes[0].set_title("Distribution des paniers")
axes[0].set_xlabel("Montant (EUR)")
axes[0].set_ylabel("Nombre de ventes")
axes[0].legend()

# Boxplot comparatif par categorie
sns.boxplot(data=ventes, x="categorie", y="montant", ax=axes[1])
axes[1].set_title("Paniers par categorie")
axes[1].set_xlabel("")
axes[1].set_ylabel("Montant (EUR)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print("Skewness (asymetrie) :", round(ventes["montant"].skew(), 2), "(>0 = queue a droite)")

## Etape 5 - Detection d'outliers (regle 1,5 x IQR + z-score)

**Regle IQR** (standard, robuste, celle des moustaches du boxplot) : toute valeur hors de `[Q1 - 1,5xIQR ; Q3 + 1,5xIQR]` est signalee. **Z-score** : `z = (x - moyenne) / ecart_type`, suspect si `|z| > 3` — mais moyenne et ecart-type sont eux-memes tires par les outliers, donc sur des donnees **tres asymetriques** on prefere l'IQR.

> Regle d'or : **on ne supprime JAMAIS un outlier en silence.** On enquete, on decide, on trace la decision.

In [ ]:
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr
outliers = ventes[(ventes["montant"] < borne_basse) | (ventes["montant"] > borne_haute)]

print(f"Bornes 'normales' : [{borne_basse:.1f} ; {borne_haute:.1f}] EUR")
print(f"Outliers IQR : {len(outliers)} ({len(outliers) / len(ventes) * 100:.1f} % des ventes)")

# Methode z-score, pour comparer
ventes["zscore"] = np.abs(stats.zscore(ventes["montant"]))
outliers_z = ventes[ventes["zscore"] > 3]
print(f"Outliers z-score (|z|>3) : {len(outliers_z)}")

# Impact sur la moyenne : avec vs sans les outliers IQR
sans = ventes[(ventes["montant"] >= borne_basse) & (ventes["montant"] <= borne_haute)]
print(f"\nMoyenne AVEC outliers : {ventes['montant'].mean():.2f} EUR")
print(f"Moyenne SANS outliers : {sans['montant'].mean():.2f} EUR")

outliers[["ville", "categorie", "produit", "quantite", "montant"]].sort_values("montant", ascending=False).head(8)

## 🎯 A toi de jouer #1 - Moyenne vs mediane de la MARGE

La colonne `marge` (benefice de chaque vente) est elle aussi asymetrique. Refais le diagnostic de l'etape 1 dessus.

In [ ]:
# TODO 1 : calcule la moyenne et la mediane de la colonne 'marge'
# moyenne_marge = ...
# mediane_marge = ...

# TODO 2 : affiche-les, puis conclus
# Indice : si moyenne > mediane => asymetrie a droite => communiquer la mediane
# print(f"Moyenne marge : {moyenne_marge:.2f} EUR | Mediane marge : {mediane_marge:.2f} EUR")

## 🎯 A toi de jouer #2 - Ecart-type par TYPE de vente

Le magasin physique et l'e-commerce ont-ils la meme regularite de paniers ? Compare-les avec le coefficient de variation.

In [ ]:
# TODO 1 : groupe les ventes par la colonne 'type' et calcule mean + std du 'montant'
# resume_type = ventes.groupby("type")["montant"].agg(["mean", "std"])

# TODO 2 : ajoute une colonne CV_% = std / mean * 100 (arrondie a 1 decimale)
# resume_type["CV_%"] = ...

# TODO 3 : quel type de vente est le plus regulier (CV le plus faible) ?
# resume_type

## 🎯 A toi de jouer #3 - Outliers d'UNE categorie au choix

Applique la regle 1,5 x IQR sur les paniers d'une seule categorie, comme a l'etape 5.

In [ ]:
# Categories disponibles (attention aux accents) :
# print(ventes["categorie"].unique())

# TODO 1 : filtre le DataFrame sur une categorie de ton choix
# sous_ensemble = ventes[ventes["categorie"] == "Mode"]

# TODO 2 : calcule q1c, q3c, iqrc, puis les bornes basse (bb) et haute (bh) avec 1,5 x IQR
# q1c, q3c = sous_ensemble["montant"].quantile([0.25, 0.75])
# iqrc = ...

# TODO 3 : compte les outliers et affiche-les
# outliers_cat = sous_ensemble[(sous_ensemble["montant"] < bb) | (sous_ensemble["montant"] > bh)]
# print(len(outliers_cat), "outliers")

## Synthese - ce que tu retiens

| Notion | Ce qu'on a fait sur `ventes_magasins.csv` | Reflexe metier |
|---|---|---|
| **Moyenne vs mediane** | Moyenne > mediane sur le `montant` -> asymetrie a droite | Sur des paniers, on communique la **mediane** (le vrai client typique) |
| **Ecart-type / CV** | Regularite par ville | Petit ecart-type = **previsible** ; le CV compare des echelles differentes |
| **Quartiles / IQR** | 50 % des ventes entre Q1 et Q3 | L'IQR est **robuste** : les extremes ne le bougent pas |
| **Histogramme + boxplot** | Forme des paniers + comparaison par categorie | L'histogramme montre la **forme**, le boxplot **compare** et repere les outliers |
| **Outliers (IQR + z-score)** | Detection des paniers anormaux et impact sur la moyenne | On **enquete** avant de decider, **jamais** de suppression silencieuse |

**Les 3 reflexes a garder :**
1. `df.info()` + `df.dtypes` -> quels types de variables ?
2. `df.describe()` -> tendance centrale + dispersion + quantiles. **Compare toujours moyenne et mediane.**
3. **Histogramme + boxplot** -> forme, asymetrie, outliers -> decision de nettoyage.

> Prochaine etape : chapitre 4 (probabilites, loi normale) puis chapitre 5 (correlation entre deux variables).